This notebook uses `autoray` to extract the computational scaling of each order of the loop cluster expansion.

In [4]:
import quimb as qu
import quimb.tensor as qtn
import autoray as ar
from autoray.experimental.complexity_tracing import (
    compute_cost_scalings,
    prime_factors,
)
import tqdm
import xyzpy as xyz

In [ ]:
def f(geom="square", C=10):

    if geom == "square":
        edges = qtn.edges_2d_square(C + 2, C + 2, cyclic=True)
        cooa = (0, 0)
        coob = (0, 1)
    elif geom == "cubic":
        edges = qtn.edges_3d_cubic(C + 2, C + 2, C + 2, cyclic=True)
        cooa = (0, 0, 0)
        coob = (0, 0, 1)

    D = 29
    p = 2
    peps = qtn.TN_from_edges_and_fill_fn(
        fill_fn=lambda shape: ar.lazy.Variable(shape, backend="numpy"),
        edges=edges,
        D=D,
        phys_dim=p,
    )
    Oij = ar.lazy.Variable((2, 2, 2, 2), backend="numpy")
    rs = tuple(
        r
        for r in peps.gen_gloops_sites(
            C, sites=[cooa, coob], grow_from="alldangle"
        )
        if len(r) == C
    )
    counts = {}
    for r in tqdm.tqdm(rs):
        psir = peps.select_any([peps.site_tag(coo) for coo in r])
        z = psir.compute_local_expectation_exact(
            {(cooa, coob): Oij}, normalized=False, optimize="random-greedy-128"
        )
        scalings = compute_cost_scalings(z, factor_map={"p": p, "D": D})
        Dscaling = max(scalings, key=lambda s: (s["D"], s["p"]))["D"]
        counts[Dscaling] = counts.get(Dscaling, 0) + 1

    return {
        "Dscaling": max(counts),
        "num_rtop": len(rs),
        "num_rtop_Dscaling": counts[max(counts)],
    }

In [6]:
ds = xyz.cultivate(
    f,
    combos={
        "geom": [
            "square",
            "cubic",
        ],
        "C": [
            2,
            4,
            5,
            6,
            7,
            8,
            # 9,
            # 10,
        ],
    },
    data_name="D_complexity_scaling.h5",
    verbosity=0,
)

In [7]:
ds

<xarray.Dataset> Size: 496B
Dimensions:            (geom: 2, C: 8)
Coordinates:
  * geom               (geom) <U6 48B 'cubic' 'square'
  * C                  (C) int64 64B 2 4 5 6 7 8 9 10
Data variables:
    Dscaling           (geom, C) float64 128B 7.0 8.0 8.0 9.0 ... 8.0 10.0 11.0
    num_rtop           (geom, C) float64 128B 1.0 4.0 16.0 ... 346.0 1.077e+03
    num_rtop_Dscaling  (geom, C) float64 128B 1.0 4.0 16.0 90.0 ... 48.0 6.0 1.0

In [9]:
for C, Ds, num in zip(
    ds.sel(geom="square")["C"].values,
    ds.sel(geom="square")["Dscaling"].values,
    ds.sel(geom="square")["num_rtop"].values,
):
    print(C, Ds, num)

2 5.0 1.0
4 6.0 2.0
5 6.0 4.0
6 8.0 15.0
7 8.0 38.0
8 8.0 126.0
9 10.0 346.0
10 11.0 1077.0


In [8]:
for C, Ds, num in zip(
    ds.sel(geom="cubic")["C"].values,
    ds.sel(geom="cubic")["Dscaling"].values,
    ds.sel(geom="cubic")["num_rtop"].values,
):
    print(C, Ds, num)

2 7.0 1.0
4 8.0 4.0
5 8.0 16.0
6 9.0 106.0
7 10.0 484.0
8 12.0 2688.0
9 12.0 13408.0
10 nan nan
